In [ ]:
# ✅ Install Required Libraries
!pip install -q transformers datasets accelerate bitsandbytes peft huggingface_hub torch

# ✅ Set CUDA Memory Optimization
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # ✅ Prevents memory fragmentation

# ✅ Log in to Hugging Face
from huggingface_hub import notebook_login
notebook_login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
import os
import torch
import gc
import json
import shutil
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
file_path = "Modified_Bengali_Review_Dataset.csv"  # Ensure this file is in your working directory
df = pd.read_csv(file_path)
df.columns = ["Reviews", "Sentiment"]  # Ensure correct column names

# Train-test split (80-20)
train_size = int(0.8 * len(df))
df_train = df[:train_size]
df_test = df[train_size:]

dataset = {
    "train": Dataset.from_pandas(df_train),
    "test": Dataset.from_pandas(df_test)
}

In [ ]:
# ✅ Function to Split Dataset into 16 Mini-Batches
def split_into_batches(dataset, num_batches=16):
    batch_size = len(dataset) // num_batches
    return [dataset.select(range(i * batch_size, (i + 1) * batch_size)) for i in range(num_batches)]

# ✅ Apply Batch Splitting
train_batches = split_into_batches(dataset["train"], num_batches=16)
test_batches = split_into_batches(dataset["test"], num_batches=16)

print(f"✅ Train set divided into {len(train_batches)} batches of {len(train_batches[0])} samples each.")
print(f"✅ Test set divided into {len(test_batches)} batches of {len(test_batches[0])} samples each.")


✅ Train set divided into 16 batches of 590 samples each.
✅ Test set divided into 16 batches of 147 samples each.


In [ ]:
print("Dataset Column Names:", dataset["train"].column_names)
print("Sample Data:", dataset["train"][0])  # Print first sample to inspect

Dataset Column Names: ['Reviews', 'Sentiment']
Sample Data: {'Reviews': ' অসাধারণ নিশো বস্ আর অমি ভাইকেও।', 'Sentiment': 0}


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "google/gemma-2b"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token  # ✅ Use EOS token as PAD token

# ✅ Few-Shot Prompt Engineering
def create_few_shot_prompt(review, label):
    prompt = f"""
নিচে কিছু রিভিউ এবং তাদের শ্রেণীবিন্যাস দেওয়া হলো:

1. "এই পণ্যটি খুবই ভালো ছিল।" → ইতিবাচক
2. "আমি একদম সন্তুষ্ট না।" → নেতিবাচক
3. "পণ্যটি ঠিকঠাক, কিন্তু ডেলিভারি দেরি হয়েছে।" → নিরপেক্ষ

এখন এই রিভিউটির শ্রেণীবিন্যাস বলুন:
"{review}"
শ্রেণীবিন্যাস: {label}
"""
    return prompt

from datasets import Dataset

# ✅ Detect and use the correct column names
def apply_few_shot_format(dataset):
    column_names = dataset.column_names  # ✅ Get available column names
    review_column = "Reviews" if "Reviews" in column_names else column_names[0]  # ✅ Default to first column if missing
    label_column = "Sentiment" if "Sentiment" in column_names else column_names[1]  # ✅ Default to second column if missing

    formatted_reviews = []
    for example in dataset:
        formatted_reviews.append({
            "text": create_few_shot_prompt(example[review_column], example[label_column]),  # ✅ Use detected column names
            "Label": example[label_column]
        })

    # ✅ Convert list back to Dataset
    return Dataset.from_list(formatted_reviews)

# ✅ Apply Fix to Dataset
dataset["train"] = apply_few_shot_format(dataset["train"])
dataset["test"] = apply_few_shot_format(dataset["test"])

# ✅ Verify Columns Again
print("After Few-Shot Formatting - Train Columns:", dataset["train"].column_names)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

After Few-Shot Formatting - Train Columns: ['text', 'Label']


In [ ]:
# ✅ Fix: Ensure Few-Shot Formatting is Applied to Train & Test Batches
train_batches = [apply_few_shot_format(batch) for batch in train_batches]
test_batches = [apply_few_shot_format(batch) for batch in test_batches]

# ✅ Verify After Applying Formatting
print("After Formatting - Train Batch Columns:", train_batches[0].column_names)
print("After Formatting - Train Batch Sample:", train_batches[0][0])


# ✅ Fix Tokenization Function
def tokenize_function(examples):
    tokens = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)
    tokens["labels"] = tokens["input_ids"].copy()  # ✅ Assign labels
    return tokens



# ✅ Print column names of first batch
print("Before Tokenization - Train Batch Columns:", train_batches[0].column_names)

# ✅ Print a sample review
print("Before Tokenization - Train Batch Sample:", train_batches[0][0])



After Formatting - Train Batch Columns: ['text', 'Label']
After Formatting - Train Batch Sample: {'text': '\nনিচে কিছু রিভিউ এবং তাদের শ্রেণীবিন্যাস দেওয়া হলো:\n\n1. "এই পণ্যটি খুবই ভালো ছিল।" → ইতিবাচক\n2. "আমি একদম সন্তুষ্ট না।" → নেতিবাচক\n3. "পণ্যটি ঠিকঠাক, কিন্তু ডেলিভারি দেরি হয়েছে।" → নিরপেক্ষ\n\nএখন এই রিভিউটির শ্রেণীবিন্যাস বলুন:\n" অসাধারণ নিশো বস্ আর অমি ভাইকেও।"\nশ্রেণীবিন্যাস: 0\n', 'Label': 0}
Before Tokenization - Train Batch Columns: ['text', 'Label']
Before Tokenization - Train Batch Sample: {'text': '\nনিচে কিছু রিভিউ এবং তাদের শ্রেণীবিন্যাস দেওয়া হলো:\n\n1. "এই পণ্যটি খুবই ভালো ছিল।" → ইতিবাচক\n2. "আমি একদম সন্তুষ্ট না।" → নেতিবাচক\n3. "পণ্যটি ঠিকঠাক, কিন্তু ডেলিভারি দেরি হয়েছে।" → নিরপেক্ষ\n\nএখন এই রিভিউটির শ্রেণীবিন্যাস বলুন:\n" অসাধারণ নিশো বস্ আর অমি ভাইকেও।"\nশ্রেণীবিন্যাস: 0\n', 'Label': 0}


In [ ]:
# ✅ Apply Tokenization to Each Mini-Batch
tokenized_train_batches = [batch.map(tokenize_function, batched=True) for batch in train_batches]
tokenized_test_batches = [batch.map(tokenize_function, batched=True) for batch in test_batches]




Map:   0%|          | 0/590 [00:00<?, ? examples/s]

Map:   0%|          | 0/590 [00:00<?, ? examples/s]

Map:   0%|          | 0/590 [00:00<?, ? examples/s]

Map:   0%|          | 0/590 [00:00<?, ? examples/s]

Map:   0%|          | 0/590 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/147 [00:00<?, ? examples/s]

Map:   0%|          | 0/147 [00:00<?, ? examples/s]

Map:   0%|          | 0/147 [00:00<?, ? examples/s]

Map:   0%|          | 0/147 [00:00<?, ? examples/s]

Map:   0%|          | 0/147 [00:00<?, ? examples/s]

In [ ]:
# ✅ Dynamically remove columns only if they exist
columns_to_remove = ["text", "Label"]  # Columns we want to remove

tokenized_train_batches = [
    batch.remove_columns([col for col in columns_to_remove if col in batch.column_names])
    for batch in tokenized_train_batches
]

tokenized_test_batches = [
    batch.remove_columns([col for col in columns_to_remove if col in batch.column_names])
    for batch in tokenized_test_batches
]


In [ ]:
print("🚀 Final Check Before Training:")
print("Final Train Batch Columns:", tokenized_train_batches[0].column_names)
print("First Sample from Train Batch:", tokenized_train_batches[0][0])


🚀 Final Check Before Training:
Final Train Batch Columns: ['input_ids', 'attention_mask', 'labels']
First Sample from Train Batch: {'input_ids': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 108, 159056, 238338, 236180, 169884, 238565, 237265, 73884, 236272, 238327, 236272, 238957, 114398, 43982, 224555, 77577, 26599, 236180, 238980, 233378, 64912, 28495, 79452, 132691, 238734, 195018, 44663, 236843, 237675, 235292, 109, 235274, 235265, 664, 237903, 237913, 28596, 238980, 28495, 58415, 126649, 237265, 236571, 237913, 81598, 66807, 237675, 162077, 106917, 235940, 235281, 13291, 106771, 96437, 180635, 238338, 236460, 108, 235284, 235265, 664, 238058, 198719, 96053, 237273, 236821, 26706, 183644, 237265, 174291, 136624, 235940, 235281, 13291, 44986, 236180, 96437, 180635, 238338, 236460, 108, 2

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import torch

# ✅ Configure 4-bit Quantization
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

# ✅ Load Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

# ✅ Apply LoRA (Low-Rank Adaptation)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

trainable params: 1,843,200 || all params: 2,508,015,616 || trainable%: 0.0735


In [ ]:
from transformers import TrainingArguments, Trainer

# ✅ Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=1e-6,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=3,
    logging_steps=10,
    fp16=True,
    report_to="none",
    remove_unused_columns=False  # ✅ Prevents Trainer from removing necessary columns
)


In [ ]:
import gc

# ✅ Training Loop Over Mini-Batches
num_epochs = 3  # ✅ Adjust if needed

for epoch in range(num_epochs):
    print(f"\n🚀 Starting Epoch {epoch + 1}/{num_epochs}")

    for batch_idx, train_batch in enumerate(tokenized_train_batches):
        print(f"📌 Training Batch {batch_idx + 1}/16 in Epoch {epoch + 1}")

        # ✅ Free GPU Memory Before Each Mini-Batch
        torch.cuda.empty_cache()
        gc.collect()

        # ✅ Create Trainer for Each Mini-Batch
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_batch,
            eval_dataset=tokenized_test_batches[batch_idx],  # ✅ Use corresponding test batch
            data_collator=None
        )

        # ✅ Train on the Mini-Batch
        trainer.train()

        # ✅ Save Model Every 4 Batches
        if batch_idx % 4 == 0:
            trainer.save_model(f"./fine_tuned_model_epoch{epoch+1}_batch{batch_idx+1}")

    print(f"✅ Completed Epoch {epoch + 1}/{num_epochs} ✅\n")



🚀 Starting Epoch 1/3
📌 Training Batch 1/16 in Epoch 1


Step,Training Loss
10,2.044600
20,2.046800
30,2.053100
40,2.021400
50,2.031800


📌 Training Batch 2/16 in Epoch 1


Step,Training Loss
10,2.029200
20,1.989000
30,1.991400
40,1.996600
50,1.983700


📌 Training Batch 3/16 in Epoch 1


...

Step,Training Loss
10,1.443300
20,1.428600
30,1.416000
40,1.454600
50,1.409900


📌 Training Batch 14/16 in Epoch 3


Step,Training Loss
10,1.400300
20,1.394500
30,1.391700
40,1.412000
50,1.402200


📌 Training Batch 15/16 in Epoch 3


Step,Training Loss
10,1.408100
20,1.377400
30,1.412500


In [ ]:
# ✅ Run Final Evaluation
print("📊 Running Final Evaluation...")
eval_results = trainer.evaluate()
print("✅ Evaluation Completed!")
print(eval_results)

# ✅ Save Everything
import json, shutil
from google.colab import files

trainer.save_model("./final_fine_tuned_model")
tokenizer.save_pretrained("./final_fine_tuned_model")

with open("results.json", "w") as f:
    json.dump(eval_results, f)

shutil.make_archive("fine_tuned_model", 'zip', "./final_fine_tuned_model")

files.download("fine_tuned_model.zip")
files.download("results.json")

print("✅ Model, Tokenizer, and Results Saved Successfully! 🚀🔥")
